In [2]:
import sqlite3
from pathlib import Path

import pandas as pd

# Zet paden relatief aan huidige directory (waar het notebook wordt uitgevoerd)
notebook_dir = Path.cwd()
source_db = notebook_dir / "GreatOutdoors_SDM.db"
inventory_csv = notebook_dir / "INVENTORY_LEVELS-data.csv"
forecast_csv = notebook_dir / "PRODUCT_FORECAST-data.csv"
target_db = notebook_dir / "GreatOutdoors_DWH.db"

In [3]:
def connect(db_path: Path) -> sqlite3.Connection:
    con = sqlite3.connect(db_path)
    con.execute("PRAGMA foreign_keys = ON")
    return con
connect(source_db)

def execute_script(con: sqlite3.Connection, sql: str) -> None:
    con.executescript(sql)
    con.commit()


def yyyymmdd(year: pd.Series, month: pd.Series, day: int = 1) -> pd.Series:
    return (year.astype(int) * 10000) + (month.astype(int) * 100) + day

In [4]:
def create_target_schema(con: sqlite3.Connection) -> None:
    schema = """
    CREATE TABLE IF NOT EXISTS dim_customer_age_group (
        customer_age_group_key INTEGER PRIMARY KEY,
        age_group_code         TEXT NOT NULL UNIQUE,
        lower_age              INTEGER,
        upper_age              INTEGER
    );

    CREATE TABLE IF NOT EXISTS dim_customer (
        customer_key                INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_code               TEXT,
        customer_headquarters_code  TEXT,
        customer_site_code          TEXT,
        company_name                TEXT,
        country                     TEXT,
        city                        TEXT,
        region                      TEXT,
        postal_zone                 TEXT,
        active_indicator            INTEGER,
        segment_name                TEXT,
        customer_type_en            TEXT,
        territory                   TEXT,
        UNIQUE(customer_code, customer_site_code)
    );

    CREATE TABLE IF NOT EXISTS dim_date (
        date_key INTEGER PRIMARY KEY,
        date     TEXT NOT NULL,
        year     INTEGER NOT NULL,
        month    INTEGER NOT NULL,
        day      INTEGER NOT NULL
    );

    CREATE TABLE IF NOT EXISTS dim_product (
        product_key           INTEGER PRIMARY KEY AUTOINCREMENT,
        product_number        INTEGER NOT NULL UNIQUE,
        introduction_date     TEXT,
        production_cost       REAL,
        margin                REAL,
        product_name          TEXT,
        product_line_en       TEXT,
        product_type_en       TEXT,
        production_cost_group TEXT,
        margin_group          TEXT,
        age                   INTEGER
    );

    CREATE TABLE IF NOT EXISTS fact_sales_demographic (
        demographic_code           TEXT PRIMARY KEY,
        customer_key               INTEGER,
        customer_age_group_key     INTEGER,
        customer_headquarters_code TEXT,
        age_group_code             TEXT,
        sales_percent              REAL,
        FOREIGN KEY(customer_key) REFERENCES dim_customer(customer_key),
        FOREIGN KEY(customer_age_group_key) REFERENCES dim_customer_age_group(customer_age_group_key)
    );

    CREATE TABLE IF NOT EXISTS fact_inventory (
        date_key        INTEGER NOT NULL,
        product_key     INTEGER NOT NULL,
        inventory_count INTEGER,
        PRIMARY KEY(date_key, product_key),
        FOREIGN KEY(date_key) REFERENCES dim_date(date_key),
        FOREIGN KEY(product_key) REFERENCES dim_product(product_key)
    );

    CREATE TABLE IF NOT EXISTS fact_product_forecast (
        date_key        INTEGER NOT NULL,
        product_key     INTEGER NOT NULL,
        expected_volume INTEGER,
        PRIMARY KEY(date_key, product_key),
        FOREIGN KEY(date_key) REFERENCES dim_date(date_key),
        FOREIGN KEY(product_key) REFERENCES dim_product(product_key)
    );
    """
    execute_script(con, schema)

In [5]:
def load_dim_customer_age_group(src: sqlite3.Connection, tgt: sqlite3.Connection) -> None:
    df = pd.read_sql_query("""
        SELECT AGE_GROUP_CODE, LOWER_AGE, UPPER_AGE
        FROM CRM_age_group
    """, src)

    df = df.rename(columns=str.lower)
    df["customer_age_group_key"] = df["age_group_code"].astype(int)
    df["lower_age"] = pd.to_numeric(df["lower_age"], errors="coerce").astype("Int64")
    df["upper_age"] = pd.to_numeric(df["upper_age"], errors="coerce").astype("Int64")

    df = df[[
        "customer_age_group_key",
        "age_group_code",
        "lower_age",
        "upper_age"
    ]].drop_duplicates()

    df.to_sql("dim_customer_age_group", tgt, if_exists="append", index=False)

In [6]:
def load_dim_customer(src: sqlite3.Connection, tgt: sqlite3.Connection) -> None:
    df = pd.read_sql_query("""
        SELECT
            c.CUSTOMER_CODE                         AS customer_code,
            c.CUSTOMER_CODEMR                       AS customer_headquarters_code,
            s.CUSTOMER_SITE_CODE                    AS customer_site_code,
            c.COMPANY_NAME                          AS company_name,
            co.COUNTRY_EN                           AS country,
            s.CITY                                  AS city,
            s.STATE                                 AS region,
            s.ZIPCODE                               AS postal_zone,
            s.ACTIVE_INDICATOR                      AS active_indicator,
            seg.SEGMENT_NAME                        AS segment_name,
            typ.CUSTOMER_TYPE_EN                    AS customer_type_en,
            terr.TERRITORY_NAME_EN                  AS territory
        FROM CRM_customer c
        LEFT JOIN CRM_customer_store s
            ON s.CUSTOMER_CODE = c.CUSTOMER_CODE
        LEFT JOIN CRM_customer_headquarters hq
            ON hq.CUSTOMER_CODEMR = c.CUSTOMER_CODEMR
        LEFT JOIN CRM_customer_segment seg
            ON seg.SEGMENT_CODE = hq.SEGMENT_CODE
           AND COALESCE(seg.LANGUAGE, 'EN') = 'EN'
        LEFT JOIN CRM_customer_type typ
            ON typ.CUSTOMER_TYPE_CODE = c.CUSTOMER_TYPE_CODE
        LEFT JOIN CRM_crm_country co
            ON co.COUNTRY_CODE = COALESCE(s.COUNTRY_CODE, hq.COUNTRY_CODE)
        LEFT JOIN CRM_sales_territory terr
            ON terr.SALES_TERRITORY_CODE = co.SALES_TERRITORY_CODE
    """, src)

    df["active_indicator"] = pd.to_numeric(
        df["active_indicator"], errors="coerce"
    ).astype("Int64")

    df = df.drop_duplicates(subset=["customer_code", "customer_site_code"])
    df.to_sql("dim_customer", tgt, if_exists="append", index=False)

In [9]:
def load_fact_sales_demographic(src: sqlite3.Connection, tgt: sqlite3.Connection) -> None:
    demo = pd.read_sql_query("""
        SELECT DEMOGRAPHIC_CODE, CUSTOMER_CODEMR, AGE_GROUP_CODE, SALES_PERCENT
        FROM CRM_sales_demographic
    """, src)

    customers = pd.read_sql_query("""
        SELECT customer_key, customer_headquarters_code
        FROM dim_customer
        WHERE customer_headquarters_code IS NOT NULL
    """, tgt).drop_duplicates(subset=["customer_headquarters_code"])

    age = pd.read_sql_query("""
        SELECT customer_age_group_key, age_group_code
        FROM dim_customer_age_group
    """, tgt)

    df = demo.rename(columns={
        "DEMOGRAPHIC_CODE": "demographic_code",
        "CUSTOMER_CODEMR": "customer_headquarters_code",
        "AGE_GROUP_CODE": "age_group_code",
        "SALES_PERCENT": "sales_percent",
    })

    df = df.merge(customers, on="customer_headquarters_code", how="left")
    df = df.merge(age, on="age_group_code", how="left")
    df["sales_percent"] = pd.to_numeric(df["sales_percent"], errors="coerce")

    df = df[[
        "demographic_code",
        "customer_key",
        "customer_age_group_key",
        "customer_headquarters_code",
        "age_group_code",
        "sales_percent"
    ]].drop_duplicates(subset=["demographic_code"])

    df.to_sql("fact_sales_demographic", tgt, if_exists="append", index=False)


In [10]:
def load_fact_inventory(tgt: sqlite3.Connection, inventory_csv: Path) -> None:
    inv = pd.read_csv(inventory_csv)

    inv = inv.rename(columns={
        "INVENTORY_YEAR": "year",
        "INVENTORY_MONTH": "month",
        "PRODUCT_NUMBER": "product_number",
        "INVENTORY_COUNT": "inventory_count",
    })

    inv["date_key"] = yyyymmdd(inv["year"], inv["month"], 1)

    products = pd.read_sql_query(
        "SELECT product_key, product_number FROM dim_product",
        tgt
    )

    products["product_number"] = products["product_number"].astype(int)

    df = inv.merge(products, on="product_number", how="inner")

    df = df[[
        "date_key",
        "product_key",
        "inventory_count"
    ]].drop_duplicates(subset=["date_key", "product_key"])

    df.to_sql("fact_inventory", tgt, if_exists="append", index=False)

In [11]:
def load_fact_product_forecast(tgt: sqlite3.Connection, forecast_csv: Path) -> None:
    fc = pd.read_csv(forecast_csv)

    fc = fc.rename(columns={
        "YEAR": "year",
        "MONTH": "month",
        "PRODUCT_NUMBER": "product_number",
        "EXPECTED_VOLUME": "expected_volume",
    })

    fc["date_key"] = yyyymmdd(fc["year"], fc["month"], 1)

    products = pd.read_sql_query(
        "SELECT product_key, product_number FROM dim_product",
        tgt
    )

    products["product_number"] = products["product_number"].astype(int)

    df = fc.merge(products, on="product_number", how="inner")

    df = df[[
        "date_key",
        "product_key",
        "expected_volume"
    ]].drop_duplicates(subset=["date_key", "product_key"])

    df.to_sql("fact_product_forecast", tgt, if_exists="append", index=False)

In [ ]:
def run_etl(source_db: Path, inventory_csv: Path, forecast_csv: Path, target_db: Path) -> None:
    if target_db.exists():
        target_db.unlink()

    with connect(source_db) as src, connect(target_db) as tgt:
        create_target_schema(tgt)

        load_dim_customer_age_group(src, tgt)
        load_dim_customer(src, tgt)


        load_fact_sales_demographic(src, tgt)
        #load_fact_inventory(tgt, inventory_csv)
        #load_fact_product_forecast(tgt, forecast_csv)

        tgt.commit()

run_etl(source_db, inventory_csv, forecast_csv, target_db)